In [66]:
# Install and import required libraries
import numpy as np
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [67]:
# Load IMDb sentiment dataset directly using its full repository name
!pip install datasets -q
from datasets import load_dataset
import pandas as pd

# Load dataset using namespace/name format
dataset = load_dataset("stanfordnlp/imdb")

# Convert to pandas DataFrame
df_train = pd.DataFrame(dataset['train'])
df_test = pd.DataFrame(dataset['test'])

# Take a subset for fast execution on Colab
df = pd.concat([df_train.sample(2000, random_state=42), df_test.sample(500, random_state=42)])
df = df.rename(columns={'text': 'review', 'label': 'sentiment'})
df.head()

,review,sentiment
6868,"Dumb is as dumb does, in this thoroughly unint...",0
24016,I dug out from my garage some old musicals and...,1
9668,After watching this movie I was honestly disap...,0
13640,This movie was nominated for best picture but ...,1
14018,Just like Al Gore shook us up with his painful...,1


In [68]:
# Download NLTK English stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [69]:
# Define text cleaning function
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<[^>]*>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

# Apply cleaning on IMDb reviews
df['cleaned_review'] = df['review'].apply(clean_text)
df[['review', 'cleaned_review']].head()

,review,cleaned_review
6868,"Dumb is as dumb does, in this thoroughly unint...",dumb dumb thoroughly uninteresting supposed bl...
24016,I dug out from my garage some old musicals and...,dug garage old musicals another one favorites ...
9668,After watching this movie I was honestly disap...,watching movie honestly disappointed actors st...
13640,This movie was nominated for best picture but ...,movie nominated best picture lost casablanca p...
14018,Just like Al Gore shook us up with his painful...,like al gore shook us painfully honest cleverl...


In [70]:
# Define text cleaning function (lowercasing, HTML removal, punctuation, stopwords)
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<[^>]*>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

# Apply preprocessing to reviews
df['cleaned_review'] = df['review'].apply(clean_text)
df[['review', 'cleaned_review']]

,review,cleaned_review
6868,"Dumb is as dumb does, in this thoroughly unint...",dumb dumb thoroughly uninteresting supposed bl...
24016,I dug out from my garage some old musicals and...,dug garage old musicals another one favorites ...
9668,After watching this movie I was honestly disap...,watching movie honestly disappointed actors st...
13640,This movie was nominated for best picture but ...,movie nominated best picture lost casablanca p...
14018,Just like Al Gore shook us up with his painful...,like al gore shook us painfully honest cleverl...
...,...,...
9774,"Pretty twisted Horror film, that has a few goo...",pretty twisted horror film good moments creepy...
4068,Master of Italian horror Dario Argento is call...,master italian horror dario argento called lot...
15582,Love In Limbo is my all-time favoirite movie. ...,love limbo alltime favoirite movie set wa hila...
18379,"I saw this when it first came out, and found i...",saw first came found work genius must confess ...


In [71]:
# Convert cleaned text to numerical vectors using TF-IDF
tfidf = TfidfVectorizer(max_features=2500)
X = tfidf.fit_transform(df['cleaned_review']).toarray()
y = df['sentiment'].values

In [72]:
# Split data into 80% train and 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Logistic Regression Model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [73]:
# Evaluate model accuracy and precision/recall
y_pred = model.predict(X_test)
print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy Score: 0.828

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.84      0.83       248
           1       0.84      0.81      0.83       252

    accuracy                           0.83       500
   macro avg       0.83      0.83      0.83       500
weighted avg       0.83      0.83      0.83       500

